In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Performance Management: A Qiskit Function by Q-CTRL Fire Opal

*ดู [API reference](https://docs.quantum.ibm.com/api/functions/q-ctrl-performance-management)*

> **Note:** Qiskit Functions เป็นฟีเจอร์ทดลองที่ให้บริการเฉพาะผู้ใช้ IBM Quantum&reg; Premium Plan, Flex Plan, และ On-Prem (ผ่าน IBM Quantum Platform API) Plan เท่านั้น อยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลงได้


<Accordion>
<AccordionItem title="Package versions">

โค้ดในหน้านี้ได้รับการพัฒนาโดยใช้ requirement ดังต่อไปนี้
เราแนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.1
qiskit-ibm-runtime~=0.45.1
```
</AccordionItem>
</Accordion>
## ภาพรวม
Fire Opal Performance Management ทำให้ทุกคนสามารถดึงผลลัพธ์ที่มีความหมายจากคอมพิวเตอร์ควอนตัมในระดับขนาดใหญ่ได้อย่างง่ายดาย โดยไม่ต้องเป็นผู้เชี่ยวชาญด้านฮาร์ดแวร์ควอนตัม เมื่อรัน Circuit กับ Fire Opal Performance Management เทคนิคการลดข้อผิดพลาดแบบ AI-driven จะถูกนำมาใช้โดยอัตโนมัติ ทำให้สามารถขยายขนาดปัญหาที่ใช้ Gate และ Qubit จำนวนมากขึ้นได้ วิธีนี้ลดจำนวน shot ที่ต้องการเพื่อให้ได้คำตอบที่ถูกต้อง โดยไม่มี overhead เพิ่มเติม ส่งผลให้ประหยัดทั้งเวลาประมวลผลและค่าใช้จ่ายได้อย่างมีนัยสำคัญ

Performance Management ลดข้อผิดพลาดและเพิ่มความน่าจะเป็นในการได้คำตอบที่ถูกต้องบนฮาร์ดแวร์ที่มีสัญญาณรบกวน กล่าวคือมันเพิ่มอัตราส่วนสัญญาณต่อสัญญาณรบกวน ภาพต่อไปนี้แสดงให้เห็นว่าความแม่นยำที่เพิ่มขึ้นจาก Performance Management ช่วยลดความจำเป็นในการใช้ shot เพิ่มเติมได้อย่างไร ในกรณีของอัลกอริทึม Quantum Fourier Transform ขนาด 10 Qubit ด้วย shot เพียง 30 ครั้ง Q-CTRL สามารถถึงเกณฑ์ความเชื่อมั่น 99% ในขณะที่ค่าเริ่มต้น (`QiskitRuntime` Sampler, `optimization_level`=3 และ `resilience_level`=1, `ibm_sherbrooke`) ต้องการถึง 170,000 shot การได้คำตอบที่ถูกต้องเร็วขึ้นช่วยประหยัดเวลาประมวลผลได้มาก

![การแสดงภาพรันไทม์ที่ปรับปรุงแล้ว](../docs/images/guides/qctrl-performance-management/achieve_more.svg)

ฟังก์ชัน Performance Management สามารถใช้กับอัลกอริทึมใดก็ได้ และใช้แทน [Qiskit Runtime primitives](/guides/primitives) มาตรฐานได้อย่างง่ายดาย เบื้องหลังมีเทคนิคการลดข้อผิดพลาดหลายอย่างทำงานร่วมกันเพื่อป้องกันข้อผิดพลาดไม่ให้เกิดขึ้นในขณะรัน เมธอด pipeline ทั้งหมดของ Fire Opal ถูกตั้งค่าล่วงหน้าและไม่ขึ้นกับอัลกอริทึม หมายความว่าคุณจะได้ประสิทธิภาพดีที่สุดทันทีโดยไม่ต้องตั้งค่าเพิ่มเติม

หากต้องการเข้าถึง Performance Management [ติดต่อ Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com)
## คำอธิบาย
Fire Opal Performance Management มีสองตัวเลือกสำหรับการประมวลผลที่คล้ายกับ Qiskit Runtime primitives ทำให้สามารถสลับไปใช้ Q-CTRL Sampler และ Estimator ได้อย่างง่ายดาย ขั้นตอนทั่วไปในการใช้ฟังก์ชัน Performance Management คือ:
1. กำหนด Circuit ของคุณ (และ operator ในกรณีของ Estimator)
2. รัน Circuit
3. ดึงผลลัพธ์

เพื่อลดสัญญาณรบกวนของฮาร์ดแวร์ Fire Opal ใช้เทคนิคการลดข้อผิดพลาดแบบ AI-driven หลายอย่างตามที่แสดงในภาพต่อไปนี้ เมื่อใช้ Fire Opal pipeline ทั้งหมดถูกทำให้เป็นอัตโนมัติอย่างสมบูรณ์โดยไม่ต้องตั้งค่าใดๆ

pipeline ของ Fire Opal ขจัดความจำเป็นต้องใช้ overhead เพิ่มเติม เช่น เวลา quantum runtime ที่เพิ่มขึ้น หรือ physical Qubit พิเศษ โปรดทราบว่าเวลาประมวลผลแบบ classical ยังคงเป็นปัจจัย (ดูส่วน [Benchmarks](#benchmarks) สำหรับการประมาณ โดย "Total time" สะท้อนทั้งการประมวลผลแบบ classical และ quantum) ต่างจาก error mitigation ที่ต้องใช้ overhead ในรูปแบบของการสุ่มตัวอย่าง การลดข้อผิดพลาดของ Fire Opal ทำงานทั้งในระดับ Gate และ pulse เพื่อจัดการกับแหล่งที่มาของสัญญาณรบกวนต่างๆ และป้องกันความน่าจะเป็นที่จะเกิดข้อผิดพลาด การป้องกันข้อผิดพลาดทำให้ไม่จำเป็นต้องใช้การประมวลผลหลังการประมวลผลที่มีค่าใช้จ่ายสูง

ภาพต่อไปนี้แสดงวิธีการลดข้อผิดพลาดที่ทำงานโดยอัตโนมัติใน Fire Opal Performance Management

![การแสดงภาพ pipeline การลดข้อผิดพลาด](../docs/images/guides/qctrl-performance-management/error_suppression.svg)

ฟังก์ชันนี้มี primitive สองตัวคือ Sampler และ Estimator โดยอินพุตและเอาต์พุตของทั้งสองขยายสเปคที่ implement สำหรับ [Qiskit Runtime V2 primitives](/guides/primitive-input-output#pubs)
## Benchmarks
ผลการ [benchmarking แบบ algorithmic ที่เผยแพร่แล้ว](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.024034) แสดงให้เห็นการปรับปรุงประสิทธิภาพอย่างมีนัยสำคัญในอัลกอริทึมต่างๆ รวมถึง Bernstein-Vazirani, quantum Fourier transform, การค้นหาของ Grover, quantum approximate optimization algorithm, และ variational quantum eigensolver ส่วนที่เหลือของส่วนนี้ให้รายละเอียดเพิ่มเติมเกี่ยวกับประเภทของอัลกอริทึมที่คุณรันได้ รวมถึงประสิทธิภาพและรันไทม์ที่คาดหวัง

การศึกษาอิสระต่อไปนี้แสดงให้เห็นว่า Performance Management ของ Q-CTRL ช่วยให้งานวิจัยเชิงอัลกอริทึมได้ในระดับที่ทำลายสถิติ:
- [Parametrized Energy-Efficient Quantum Kernels for Network Service Fault Diagnosis](https://arxiv.org/abs/2405.09724v1) - quantum kernel learning สูงสุด 50 Qubit
- [Tensor-based quantum phase difference estimation for large-scale demonstration](https://arxiv.org/abs/2408.04946) - quantum phase estimation สูงสุด 33 Qubit
- [Hierarchical Learning for Quantum ML: Novel Training Technique for Large-Scale Variational Quantum Circuits](https://arxiv.org/abs/2311.12929) - quantum data loading สูงสุด 21 Qubit

ตารางต่อไปนี้ให้แนวทางคร่าวๆ เกี่ยวกับความแม่นยำและรันไทม์จากการรัน benchmarking ก่อนหน้าบน `ibm_fez` ประสิทธิภาพบนอุปกรณ์อื่นอาจแตกต่างกัน เวลาการใช้งานอ้างอิงจากสมมติฐาน 10,000 shot ต่อ Circuit "จำนวน Qubit" ที่ระบุไม่ใช่ข้อจำกัดแบบเด็ดขาด แต่แสดงถึงเกณฑ์คร่าวๆ ที่คาดว่าจะได้ความแม่นยำของคำตอบที่สม่ำเสมออย่างมาก ปัญหาที่มีขนาดใหญ่กว่าได้รับการแก้ไขสำเร็จแล้ว และขอแนะนำให้ทดสอบเกินขีดจำกัดเหล่านี้

| ตัวอย่าง    | จำนวน Qubit | ความแม่นยำ | ตัวชี้วัดความแม่นยำ | เวลารวม (วินาที) | การใช้งาน Runtime (วินาที) | Primitive (Mode) |
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |------------- |
| Bernstein–Vazirani  |  50Q    | 100%  | Success Rate (เปอร์เซ็นต์ของการรันที่คำตอบที่ถูกต้องมีจำนวนนับสูงสุด)     | 10    | 8         | Sampler |
| Quantum Fourier Transform | 30Q              | 100% | Success Rate (เปอร์เซ็นต์ของการรันที่คำตอบที่ถูกต้องมีจำนวนนับสูงสุด)      | 10    | 8        | Sampler |
| Quantum Phase Estimation  | 30Q   | 99.9998%  | ความแม่นยำของมุมที่พบ: `1- abs(real_angle - angle_found)/pi`  | 10  | 8  | Sampler |
| Quantum simulation: Ising model (15 steps) | 20Q  | 99.775%   |  $A$ (นิยามด้านล่าง)  |  60 (ต่อ step)  | 15 (ต่อ step)   | Estimator |
| Quantum simulation 2: molecular dynamics (20 time points) | 34Q  |  96.78%  |  $A_{mean}$ (นิยามด้านล่าง)   | 10 (ต่อจุดเวลา)    | 6 (ต่อจุดเวลา)  | Estimator |

การกำหนดความแม่นยำของการวัด expectation value - เมตริก $A$ นิยามดังนี้:
$$
A = 1 - \frac{|\epsilon^{ideal} - \epsilon^{meas}|}{\epsilon^{ideal}_{max} - \epsilon^{ideal}_{min}},
$$
โดยที่ $ \epsilon^{ideal} $ = expectation value ในอุดมคติ, $ \epsilon^{meas} $ = expectation value ที่วัดได้, $\epsilon^{ideal}_{max} $ = ค่าสูงสุดในอุดมคติ, และ $\epsilon^{ideal}_{min}$ = ค่าต่ำสุดในอุดมคติ $A_{mean}$ คือค่าเฉลี่ยของ $A$ จากการวัดหลายครั้ง

เมตริกนี้ถูกใช้เพราะไม่เปลี่ยนแปลงเมื่อมีการเลื่อนหรือการปรับขนาดทั่วโลกในช่วงของค่าที่เป็นไปได้ กล่าวคือไม่ว่าคุณจะเลื่อนช่วงของ expectation value ที่เป็นไปได้ขึ้นหรือลง หรือเพิ่มการกระจาย ค่าของ $A$ ก็ควรคงที่
## เริ่มต้นใช้งาน
Fire Opal Performance Management ใช้ Qiskit v`2.0.0` ซึ่งเป็นเวอร์ชันที่แนะนำ เวอร์ชันที่รองรับคือ Qiskit >=v`2.0.0`
ยืนยันตัวตนโดยใช้ [IBM Quantum Platform API key](http://quantum.cloud.ibm.com/) และเลือก Qiskit Function ดังนี้ (snippet นี้สมมติว่าคุณได้ [บันทึกบัญชีของคุณ](/guides/functions#install-qiskit-functions-catalog-client) ไว้ในสภาพแวดล้อมท้องถิ่นแล้ว)

In [1]:
# This cell is hidden from users. It hides the "...You have imported samplomatic..." warning.
import warnings

warnings.filterwarnings("ignore", message=".*You have imported samplomatic*")

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [3]:
# Access Function
perf_mgmt = catalog.load("q-ctrl/performance-management")

<Admonition type="note" title="Does this function support all IBM backends?">
If you want to use a backend that this function does not currently support, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) to add support.
</Admonition>

## Estimator primitive

### Estimator example

Use Fire Opal Performance Management's Estimator primitive to determine the expectation value of a single circuit-observable pair.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the `numpy` package to run this example. You can install this package by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [4]:
# %pip install numpy

**2. รัน Circuit**

รัน Circuit และกำหนด Backend และจำนวน shot ตามต้องการ

In [5]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian, SparsePauliOp

n_qubits = 50

# Generate a random circuit
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

# Define observables as a string
observable = SparsePauliOp("Z" * n_qubits)

In [6]:
# Create PUB tuple
estimator_pubs = [(circuit, observable)]

คุณสามารถใช้ [Qiskit Serverless APIs](/guides/serverless) ที่คุ้นเคยเพื่อตรวจสอบสถานะของ Qiskit Function workload:

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [8]:
# Run the circuit using Estimator
qctrl_estimator_job = perf_mgmt.run(
    primitive="estimator",
    pubs=estimator_pubs,
    backend_name=backend_name,
)

**3. ดึงผลลัพธ์**

In [9]:
qctrl_estimator_job.status()

'QUEUED'

ผลลัพธ์มีรูปแบบเดียวกับ [Estimator result](/guides/estimator-input-output):

In [10]:
# Retrieve the counts from the result list
result = qctrl_estimator_job.result()

The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [22]:
import numpy

result_str = str(result)

with numpy.printoptions(threshold=200):
    print(
        f"The result of the submitted job had {len(result)} PUB "
        f"and has a value:\n {result[0]}\n"
    )

print("The associated PubResult of this job has the following DataBins:")
print(f"{result[0].data}\n")

print(f"And this DataBin has attributes: {result[0].data.keys()}")

print("The expectation values measured from this PUB are:")
print(f"{result[0].data.evs}")

The result of the submitted job had 1 PUB
The result of the submitted job had 1 PUB and has a value:
 PubResult(data=DataBin(evs=0.0195, stds=0.9998098569228051), metadata={'precision': None})

The associated PubResult of this job has the following DataBins:
DataBin(evs=0.0195, stds=0.9998098569228051)

And this DataBin has attributes: dict_keys(['evs', 'stds'])
The expectation values measured from this PUB are:
0.0195


## Sampler primitive
### Sampler example
ใช้ Sampler primitive ของ Fire Opal Performance Management เพื่อรัน Bernstein–Vazirani circuit อัลกอริทึมนี้ใช้สำหรับค้นหา hidden string จาก output ของฟังก์ชัน black box และเป็นอัลกอริทึม benchmarking ที่นิยมใช้กันเพราะมีคำตอบที่ถูกต้องเพียงคำตอบเดียว
**1. สร้าง Circuit**

กำหนดคำตอบที่ถูกต้องของอัลกอริทึม ซึ่งก็คือ hidden bitstring และ Bernstein–Vazirani circuit ปรับความกว้างของ Circuit ได้โดยแค่เปลี่ยนค่า `circuit_width`

In [12]:
import qiskit

circuit_width = 35
hidden_bitstring = "1" * circuit_width

# Create circuit, reserving one qubit for BV oracle
bv_circuit = qiskit.QuantumCircuit(circuit_width + 1, circuit_width)
bv_circuit.x(circuit_width)
bv_circuit.h(range(circuit_width + 1))
for input_qubit, bit in enumerate(reversed(hidden_bitstring)):
    if bit == "1":
        bv_circuit.cx(input_qubit, circuit_width)
bv_circuit.barrier()
bv_circuit.h(range(circuit_width + 1))
bv_circuit.barrier()
for input_qubit in range(circuit_width):
    bv_circuit.measure(input_qubit, input_qubit)

# Create PUB tuple
sampler_pubs = [(bv_circuit,)]

**2. รัน Circuit**

รัน Circuit และกำหนด backend กับจำนวน shots ได้ตามต้องการ

In [13]:
# Run the circuit using Sampler
qctrl_sampler_job = perf_mgmt.run(
    primitive="sampler",
    pubs=sampler_pubs,
    backend_name=backend_name,
)

ตรวจสอบ[สถานะ](/guides/functions#check-job-status)ของ Qiskit Function workload หรือดึง[ผลลัพธ์](/guides/functions#retrieve-results)ได้ดังนี้:

In [14]:
# Print the ID so you can use it later, if necessary
print(qctrl_sampler_job.job_id)

qctrl_sampler_job.status()

60fe2fa1-a860-43e4-8615-c6ac4180f93b


'QUEUED'

**3. Retrieve the result**

In [15]:
# Retrieve the job results
sampler_result = qctrl_sampler_job.result()

In [16]:
# Get results for the first (and only) PUB
pub_result = sampler_result[0]
counts = pub_result.data.c.get_counts()

print("Counts for the meas output register (limited to 30 results):")
for i, (bitstring, count) in enumerate(counts.items()):
    if i >= 50:
        print(f"  ... ({len(counts) - 30} more items)")
        break
    print(f"  {bitstring}: {count}")

Counts for the meas output register (limited to 30 results):
  11111111111111111111111111111111111: 1661
  11111111111111111111111111110111111: 60
  11111111111111111111111111111101111: 54
  11111111111111111111111111111110111: 54
  11111111111111011111111111111111111: 46
  11111111111111111110111111111111111: 44
  11111111111111111111111101111111111: 42
  11111111111111111111111110111111111: 42
  11111111111111110111111111111111111: 41
  11111111111111111111111111111111101: 39
  11111111111111111111101111111111111: 38
  11111111111111111111110111111111111: 38
  11111111111111111111111111101111111: 37
  11111111111111111111111111111111110: 36
  11111111111110111111111111111111111: 35
  11111111111111111111111111111011111: 32
  11111111111111101111111111111111111: 32
  01111111111111111111111111111111111: 27
  11111111111111111011111111111111111: 23
  11111111101111111111111111111111111: 22
  11111111111111111111111111111111011: 21
  11111111011111111111111111111111111: 20
  00000000000

**3. Plot the top bitstrings**

Plot the bitstring with the highest counts to see if the hidden bitstring was the mode.

In [17]:
import matplotlib.pyplot as plt


def plot_top_bitstrings(counts_dict, hidden_bitstring=None):
    # Sort and take the top 100 bitstrings
    top_100 = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)[
        :100
    ]
    if not top_100:
        print("No bitstrings found in the input dictionary.")
        return

    # Unzip the bitstrings and their counts
    bitstrings, counts = zip(*top_100)

    # Assign colors: purple if the bitstring matches hidden_bitstring,
    # otherwise gray
    colors = [
        "#680CE9" if bit == hidden_bitstring else "gray" for bit in bitstrings
    ]

    # Create the bar plot
    plt.figure(figsize=(15, 8))
    plt.bar(
        range(len(bitstrings)), counts, tick_label=bitstrings, color=colors
    )

    # Rotate the bitstrings for better readability
    plt.xticks(rotation=90, fontsize=8)
    plt.xlabel("Bitstrings")
    plt.ylabel("Counts")
    plt.title("Top 100 Bitstrings by Counts")

    # Show the plot
    plt.tight_layout()
    plt.show()

The hidden bitstring is highlighted in purple, and it should be the bitstring with the highest number of counts.

In [18]:
plot_top_bitstrings(counts, hidden_bitstring)

<Image src="../docs/images/guides/q-ctrl-performance-management/extracted-outputs/8106d906-0.avif" alt="Output of the previous code cell" />

**3. พล็อต bitstring ยอดนิยม**

พล็อต bitstring ที่มี count สูงสุดเพื่อดูว่า hidden bitstring เป็น mode หรือเปล่า